In [1]:
import numpy as np
import pandas as pd
from datetime import datetime as dt
from openbb import obb

import os

import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

import OpenBBConvenienceFunctions as obb_cf
import PlottingFunctions as plot_func

from sklearn.preprocessing import StandardScaler

In [2]:
from API_Keys import FMP_API
from File_Paths import raw_data_filepath

In [3]:
start = '2010-01-01'
end = '2025-12-31'

In [4]:
date_list = pd.date_range(start=start, end=end, freq='B').strftime("%Y-%m-%d").to_list()

### FUTURES VOLUME

In [ ]:
# ES front-month continuous futures volume from yfinance
es_hist = obb.derivatives.futures.historical('ES', provider='yfinance',
                                              start_date=start, end_date=end)
es_df = es_hist.to_df()
es_df

In [ ]:
plot_func.plot_line(es_df[['volume']], title='S&P 500 E-mini Futures Volume (ES) | 2023 to 2025',
                    ylabel='Volume', figsize=(11,4), dark=False);

In [ ]:
plot_func.plot_histogram(es_df[['volume']], bins=100,
                         title='Distribution of ES Futures Volume',
                         figsize=(11,4), dark=False, kde=True);

### CREDIT SPREAD

In [ ]:
# ICE BofA US High Yield Option-Adjusted Spread via FRED
# Requires: fred_api_key  (https://fred.stlouisfed.org)
# obb.user.credentials(fred_api_key='YOUR_KEY')
hy_spread = obb.economy.fred_series(symbol='BAMLH0A0HYM2',
                                    start_date=start, end_date=end, provider='fred')
hy_spread_df = hy_spread.to_df()
hy_spread_df

In [ ]:
plot_func.plot_line(hy_spread_df[['value']], title='US High Yield OAS (ICE BofA) | 2010 to 2025',
                    ylabel='Spread (bps)', figsize=(11,4), dark=False);

In [ ]:
plot_func.plot_histogram(hy_spread_df[['value']], bins=100,
                         title='Distribution of US HY Credit Spread',
                         figsize=(11,4), dark=False, kde=True);

### CORPORATE BOND PRICE MOVEMENTS

In [ ]:
# ICE BofA US Corporate Bond Total Return Index via FRED
# Requires: fred_api_key  (https://fred.stlouisfed.org)
# obb.user.credentials(fred_api_key='YOUR_KEY')
corp_bond = obb.economy.fred_series(symbol='BAMLCC0A0CMTRIV',
                                    start_date=start, end_date=end, provider='fred')
corp_bond_df = corp_bond.to_df()
corp_bond_df = corp_bond_df.rename(columns={'value': 'total_return_index'})
# Compute daily log returns
corp_bond_df['log_return'] = np.log(corp_bond_df['total_return_index']
                                    / corp_bond_df['total_return_index'].shift(1))
corp_bond_df.dropna(inplace=True)
corp_bond_df

In [ ]:
plot_func.plot_line(corp_bond_df[['total_return_index']],
                    title='US IG Corporate Bond Total Return Index | 2010 to 2025',
                    figsize=(11,4), dark=False);

In [ ]:
plot_func.plot_histogram(corp_bond_df[['log_return']], bins=100,
                         title='Distribution of US IG Corporate Bond Log Returns',
                         figsize=(11,4), dark=False, kde=True);

In [ ]:
# S&P 500 Broad Based Stock Index COT - code 13874A
# To discover other codes: obb.regulators.cftc.cot_search(query='S&P 500').to_df()
cot_raw = obb.regulators.cftc.cot('13874A', provider='cftc')
cot_df = cot_raw.to_df()
cot_df.index = pd.to_datetime(cot_df.index)

cot_pos = cot_df[[
    'open_interest_all',
    'noncomm_positions_long_all',
    'noncomm_positions_short_all',
    'comm_positions_long_all',
    'comm_positions_short_all',
]].sort_index()
cot_pos

In [ ]:
plot_func.plot_line(
    cot_pos[['noncomm_positions_long_all', 'noncomm_positions_short_all']],
    title='COT: S&P 500 Non-Commercial (Speculator) Positions | 1997 to 2025',
    ylabel='Contracts', figsize=(11,4), dark=False
);

In [ ]:
cot_pos = cot_pos.copy()
cot_pos['noncomm_net'] = cot_pos['noncomm_positions_long_all'] - cot_pos['noncomm_positions_short_all']
plot_func.plot_line(cot_pos[['noncomm_net']],
                    title='COT: Net Non-Commercial (Speculator) Position | 1997 to 2025',
                    ylabel='Net Contracts', figsize=(11,4), dark=False);

In [ ]:
plot_func.plot_histogram(cot_pos[['noncomm_net']], bins=80,
                         title='Distribution of Net Speculator Position (COT)',
                         figsize=(11,4), dark=False, kde=True);

### CPI

In [ ]:
cpi_raw = obb.economy.cpi(country='united_states', start_date=start, end_date=end, provider='oecd')
cpi_df = cpi_raw.to_df()
if 'expenditure' in cpi_df.columns:
    cpi_df = cpi_df[cpi_df['expenditure'] == 'Total'].drop(columns=['country', 'expenditure'])
cpi_df

In [ ]:
plot_func.plot_line(cpi_df[['value']], title='US CPI YoY % Change (OECD) | 2010 to 2025',
                    ylabel='YoY %', figsize=(11,4), dark=False);

In [ ]:
plot_func.plot_histogram(cpi_df[['value']], bins=60,
                         title='Distribution of US CPI YoY %',
                         figsize=(11,4), dark=False, kde=True);

### FED FUNDS RATE

In [ ]:
effr_raw = obb.fixedincome.rate.effr(start_date=start, end_date=end, provider='federal_reserve')
effr_df = effr_raw.to_df()
effr_df

In [ ]:
plot_func.plot_line(effr_df[['rate']], title='Effective Federal Funds Rate | 2010 to 2025',
                    ylabel='Rate (%)', figsize=(11,4), dark=False);

In [ ]:
plot_func.plot_histogram(effr_df[['rate']], bins=80,
                         title='Distribution of Effective Federal Funds Rate',
                         figsize=(11,4), dark=False, kde=True);

### UNEMPLOYMENT

In [ ]:
unemp_raw = obb.economy.unemployment(country='united_states', start_date=start, end_date=end, provider='oecd')
unemp_df = unemp_raw.to_df().drop(columns=['country'], errors='ignore')
unemp_df

In [ ]:
plot_func.plot_line(unemp_df[['value']], title='US Unemployment Rate (OECD) | 2010 to 2025',
                    ylabel='Unemployment Rate (%)', figsize=(11,4), dark=False);

In [ ]:
plot_func.plot_histogram(unemp_df[['value']], bins=60,
                         title='Distribution of US Unemployment Rate',
                         figsize=(11,4), dark=False, kde=True);

### PMI

In [ ]:
# ISM Manufacturing PMI via FRED (series: NAPM)
# Requires: fred_api_key  (https://fred.stlouisfed.org)
# obb.user.credentials(fred_api_key='YOUR_KEY')
pmi_raw = obb.economy.fred_series(symbol='NAPM', start_date=start, end_date=end, provider='fred')
pmi_df = pmi_raw.to_df()
pmi_df

In [ ]:
plot_func.plot_line(pmi_df[['value']], title='ISM Manufacturing PMI | 2010 to 2025',
                    ylabel='PMI Index', figsize=(11,4), dark=False);

In [ ]:
plot_func.plot_histogram(pmi_df[['value']], bins=60,
                         title='Distribution of ISM Manufacturing PMI',
                         figsize=(11,4), dark=False, kde=True);

### ECONOMIC CALENDAR

In [ ]:
# Economic calendar via FRED (release dates)
# Requires: fred_api_key  (https://fred.stlouisfed.org)
# obb.user.credentials(fred_api_key='YOUR_KEY')
#
# Alternatively use provider='fmp' with an FMP API key:
# obb.user.credentials(fmp_api_key='YOUR_KEY')
econ_cal = obb.economy.calendar(start_date='2024-01-01', end_date='2024-12-31', provider='fred')
econ_cal_df = econ_cal.to_df()
econ_cal_df

In [ ]:
econ_cal_df.index = pd.to_datetime(econ_cal_df.index)
events_per_month = econ_cal_df.resample('ME').size().to_frame('event_count')
plot_func.plot_bar(events_per_month, title='Economic Calendar Events Per Month | 2024',
                   ylabel='Event Count', figsize=(11,4), dark=False);

### TIMESTAMP OF EVENTS

In [ ]:
# Economic calendar with intraday timestamps via FMP
# Requires: fmp_api_key  (https://financialmodelingprep.com)
# obb.user.credentials(fmp_api_key='YOUR_KEY')
econ_cal_fmp = obb.economy.calendar(start_date='2024-01-01', end_date='2024-01-31', provider='fmp')
econ_cal_fmp_df = econ_cal_fmp.to_df()
# 'date' index contains full datetime (YYYY-MM-DD HH:MM:SS)
econ_cal_fmp_df

In [ ]:
econ_cal_fmp_df.index = pd.to_datetime(econ_cal_fmp_df.index)
release_hours = econ_cal_fmp_df.index.hour
release_hours_series = pd.Series(release_hours).value_counts().sort_index().to_frame('count')
plot_func.plot_bar(release_hours_series, title='Economic Releases by Hour of Day (EST) | Jan 2024',
                   xlabel='Hour (24h)', ylabel='Count', figsize=(11,4), dark=False);

### COMPOSITE LEADING INDICATOR

In [ ]:
cli_raw = obb.economy.composite_leading_indicator(
    country='united_states', start_date=start, end_date=end, provider='oecd'
)
cli_df = cli_raw.to_df().drop(columns=['country'], errors='ignore')
cli_df

In [ ]:
plot_func.plot_line(cli_df[['value']], title='US OECD Composite Leading Indicator | 2010 to 2025',
                    ylabel='CLI Index', figsize=(11,4), dark=False);

In [ ]:
plot_func.plot_histogram(cli_df[['value']], bins=60,
                         title='Distribution of US Composite Leading Indicator',
                         figsize=(11,4), dark=False, kde=True);

### COMPANY NEWS

In [ ]:
# Company news via yfinance (no API key required)
# For full historical coverage use: benzinga, fmp, intrinio, or tiingo (require API keys)
news_raw = obb.news.company(
    symbols='AAPL,MSFT,NVDA,AMZN,META',
    start_date='2024-01-01', end_date='2024-12-31',
    provider='yfinance'
)
news_df = news_raw.to_df()
news_df

In [ ]:
# Article count per symbol
if 'symbols' in news_df.columns:
    news_counts = news_df['symbols'].explode().value_counts().to_frame('article_count')
elif 'symbol' in news_df.columns:
    news_counts = news_df['symbol'].value_counts().to_frame('article_count')
else:
    news_df.index = pd.to_datetime(news_df.index)
    news_counts = news_df.resample('ME').size().to_frame('article_count')
plot_func.plot_bar(news_counts, title='News Article Count by Symbol | 2024',
                   ylabel='Article Count', figsize=(11,4), dark=False, horizontal=True);

### CALENDAR EARNINGS

In [ ]:
# Earnings calendar
# Requires: fmp_api_key  (https://financialmodelingprep.com)
# obb.user.credentials(fmp_api_key='YOUR_KEY')
earnings_raw = obb.equity.calendar.earnings(
    start_date='2024-01-01', end_date='2024-12-31', provider='fmp'
)
earnings_df = earnings_raw.to_df()
earnings_df

In [ ]:
earnings_df.index = pd.to_datetime(earnings_df.index)
earnings_per_week = earnings_df.resample('W').size().to_frame('earnings_count')
plot_func.plot_line(earnings_per_week, title='Earnings Releases Per Week | 2024',
                    ylabel='Count', figsize=(11,4), dark=False);

### CALENDAR EVENTS

In [ ]:
# Corporate events calendar (dividends, splits, M&A, etc.)
# Requires: fmp_api_key  (https://financialmodelingprep.com)
# obb.user.credentials(fmp_api_key='YOUR_KEY')
events_raw = obb.equity.calendar.events(
    start_date='2024-01-01', end_date='2024-12-31', provider='fmp'
)
events_df = events_raw.to_df()
events_df

In [ ]:
events_df.index = pd.to_datetime(events_df.index)
type_col = next((c for c in ['event_type', 'type', 'action'] if c in events_df.columns), None)
if type_col:
    event_counts = events_df[type_col].value_counts().to_frame('count')
    plot_func.plot_bar(event_counts, title='Corporate Events by Type | 2024',
                       ylabel='Count', figsize=(11,4), dark=False, horizontal=True)
else:
    events_per_month = events_df.resample('ME').size().to_frame('event_count')
    plot_func.plot_bar(events_per_month, title='Corporate Events Per Month | 2024',
                       ylabel='Event Count', figsize=(11,4), dark=False);

### FAMA FRENCH FACTORS

In [ ]:
# Fama-French factors are not directly available via OpenBB.
# Standard approach: pandas_datareader pulls from Ken French's data library.
import pandas_datareader.data as web

ff5 = web.DataReader('F-F_Research_Data_5_Factors_2x3', 'famafrench',
                     start=start, end=end)[0]
ff5.index = pd.to_datetime(ff5.index.to_timestamp())
ff5 = ff5 / 100  # convert percent to decimal
ff5.columns = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']
ff5

In [ ]:
plot_func.plot_line(ff5[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']],
                    title='Fama-French 5 Factors (Monthly) | 2010 to 2025',
                    ylabel='Return', figsize=(11,5), dark=False);

In [ ]:
plot_func.plot_histogram(
    {col: ff5[col].dropna().values for col in ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']},
    bins=60, title='Distribution of FF5 Factor Returns',
    figsize=(11,4), dark=False, kde=False
);

### EQUITY SHORT INTEREST

In [ ]:
# OpenBB exposes SEC Fails-to-Deliver as the closest free proxy for short interest.
# True FINRA short interest requires a paid data provider.
ftd_spy = obb.equity.shorts.fails_to_deliver(
    symbol='SPY', start_date=start, end_date=end, provider='sec'
)
ftd_spy_df = ftd_spy.to_df()
ftd_spy_df.index = pd.to_datetime(ftd_spy_df.index)
ftd_spy_df

In [ ]:
plot_func.plot_line(ftd_spy_df[['quantity']],
                    title='SPY Fails-to-Deliver (Short Interest Proxy) | 2010 to 2025',
                    ylabel='FTD Shares', figsize=(11,4), dark=False);

In [ ]:
plot_func.plot_histogram(ftd_spy_df[['quantity']], bins=80,
                         title='Distribution of SPY Fails-to-Deliver',
                         figsize=(11,4), dark=False, kde=True);

### EQUITY FTD (FAIL TO DELIVER)

In [ ]:
# SEC Fails-to-Deliver for individual equities
ftd_aapl = obb.equity.shorts.fails_to_deliver(
    symbol='AAPL', start_date=start, end_date=end, provider='sec'
)
ftd_aapl_df = ftd_aapl.to_df()
ftd_aapl_df.index = pd.to_datetime(ftd_aapl_df.index)
ftd_aapl_df

In [ ]:
plot_func.plot_line(ftd_aapl_df[['quantity']], title='AAPL Fails-to-Deliver | 2010 to 2025',
                    ylabel='FTD Shares', figsize=(11,4), dark=False);

In [ ]:
plot_func.plot_histogram(ftd_aapl_df[['quantity']], bins=80,
                         title='Distribution of AAPL Fails-to-Deliver',
                         figsize=(11,4), dark=False, kde=True);

### FAMA FRENCH BREAKPOINTS

In [ ]:
# Fama-French breakpoints are not available in OpenBB.
# Source: Ken French's data library via pandas_datareader.
ff_bp = web.DataReader('ME_Breakpoints', 'famafrench', start=start, end=end)[0]
ff_bp.index = pd.to_datetime(ff_bp.index.to_timestamp())
ff_bp.columns = ff_bp.columns.astype(str)
ff_bp

In [ ]:
plot_func.plot_line(ff_bp, title='Fama-French Size (ME) Breakpoints (Monthly) | 2010 to 2025',
                    ylabel='Market Equity ($M)', figsize=(11,5), dark=False);

In [ ]:
ff_beme_bp = web.DataReader('BE-ME_Breakpoints', 'famafrench', start=start, end=end)[0]
ff_beme_bp.index = pd.to_datetime(ff_beme_bp.index.to_timestamp())
ff_beme_bp.columns = ff_beme_bp.columns.astype(str)
plot_func.plot_line(ff_beme_bp,
                    title='Fama-French Value (BE/ME) Breakpoints (Annual) | 2010 to 2025',
                    ylabel='BE/ME Ratio', figsize=(11,5), dark=False);

### YIELD CURVE STEEPNESS

In [ ]:
# Current yield curve snapshot from econdb
yc_raw = obb.fixedincome.government.yield_curve(country='united_states', provider='econdb')
yc_df = yc_raw.to_df().sort_values('maturity_years')
yc_df

In [ ]:
plot_func.plot_line(yc_df.set_index('maturity_years')[['rate']],
                    title='US Treasury Yield Curve (econdb) | Current Snapshot',
                    xlabel='Maturity (Years)', ylabel='Yield (%)',
                    figsize=(11,4), dark=False, markers=True);

In [ ]:
# 10Y-2Y Yield Spread (steepness) time series via FRED
# Requires: fred_api_key  (https://fred.stlouisfed.org)
# obb.user.credentials(fred_api_key='YOUR_KEY')
yc_spread = obb.economy.fred_series(symbol='T10Y2Y', start_date=start, end_date=end, provider='fred')
yc_spread_df = yc_spread.to_df()
plot_func.plot_line(yc_spread_df[['value']],
                    title='Yield Curve Steepness: 10Y-2Y Spread | 2010 to 2025',
                    ylabel='Spread (%)', figsize=(11,4), dark=False);

In [ ]:
plot_func.plot_histogram(yc_spread_df[['value']], bins=80,
                         title='Distribution of 10Y-2Y Yield Curve Spread',
                         figsize=(11,4), dark=False, kde=True);